# PACSUN2603 Data
This notebook unifies everything into one dataframe for each experiment to make plotting easier later.

## Geographic Data

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import io
import pytz
import py
import matplotlib.pyplot as plt
from pvlib.solarposition import get_solarposition

# Download file:
# r2r.get_cruise_nav("RR2402")

# Read the CSV data directly from the file.
# The 'comment='#' will automatically skip lines starting with '#'.
gdf = pd.read_csv("ISAC_PACSUN2603.csv", comment='#')

# Convert 'iso_time' column to datetime objects
gdf['Datetime'] = pd.to_datetime(gdf['DeviceDateTime']).dt.tz_localize('Pacific/Tongatapu')

# Sort the DataFrame by time to ensure correct animation order
gdf = gdf.sort_values(by='Datetime')
gdf.head()

/home/pliny/.local/lib/python3.9/site-packages/numpy/core/getlimits.py:542: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)
/tmp/ipykernel_5512/3195970324.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdf['Datetime'] = pd.to_datetime(gdf['DeviceDateTime']).dt.tz_localize('Pacific/Tongatapu')


,DataId,CommId,DeviceName,DeviceDateTime,AgeInSeconds,BatteryVoltage,GpsQuality,Latitude,Longitude,SubmergedBoolean,Temperature0cm,Unnamed: 11,station,Datetime
386,66038667,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:45,63906,7.5,3,33.204360,-117.306144,0,-0.05,NaN,387,2026-03-05 17:45:00+13:00
385,66038669,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:50,64206,12.0,3,33.204357,-117.306149,0,NaN,NaN,386,2026-03-05 17:50:00+13:00
384,66038680,3.010000e+14,USC/MTEL-UT-0001,3/5/26 17:55,64507,12.0,3,33.204357,-117.306141,0,NaN,NaN,385,2026-03-05 17:55:00+13:00
383,66038795,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:00,64808,12.0,3,33.204363,-117.306149,0,NaN,NaN,384,2026-03-05 18:00:00+13:00
382,66038808,3.010000e+14,USC/MTEL-UT-0001,3/5/26 18:05,65108,12.0,3,33.204357,-117.306144,0,NaN,NaN,383,2026-03-05 18:05:00+13:00


## ISAC1

In [2]:
# Unify our tables....
dfs = []

for filename in ["ISAC1_top"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']
    df['backscatter_median'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_median'] = df['ambient'].rolling(window=11, center=True).median()
    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Pacific/Tongatapu') #
    df = df.sort_values(by="unixTime")

    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
df = pd.concat(dfs, ignore_index=True)

# Localize datetime to Tonga, assuming computer RTC is in Pacific Time:
# df['Datetime'] = df['Datetime'].dt.tz_localize('Pacific/Tongatapu')

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
gdf = gdf.sort_values('Datetime')
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')
# Save dataframe
df.to_csv('ISAC1.csv', index=False)

OpenOBS SN:123
Sun is highest at: 2026-03-19 22:32:55+13:00


## ISAC2

In [3]:
# Unify our tables....
dfs = []

for filename in ["ISAC2_mid_123"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']
    df['backscatter_median'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_median'] = df['ambient'].rolling(window=11, center=True).median()
    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Pacific/Tongatapu') #
    df = df.sort_values(by="unixTime")

    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
df = pd.concat(dfs, ignore_index=True)

# Localize datetime to Tonga, assuming computer RTC is in Pacific Time:
# df['Datetime'] = df['Datetime'].dt.tz_localize('Pacific/Tongatapu')

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
gdf = gdf.sort_values('Datetime')
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC2.csv', index=False)

OpenOBS SN:123
Sun is highest at: 2026-03-21 12:47:12+13:00


## ISAC 3


In [ ]:
# Unify our tables....
dfs = []

for filename in ["ISAC3_top_123", "ISAC3_mid_117", "ISAC3_mid_113"]:
    df = pd.read_csv(filename+".TXT", skiprows = 4)
    df.columns = ['unixTime','background','ambient', 'backscatter', 'pressure','waterTemp','battery']
    df['backscatter_median'] = df['backscatter'].rolling(window=20, center=True).median()
    df['ambient_median'] = df['ambient'].rolling(window=11, center=True).median()
    df['Datetime'] = pd.to_datetime(df['unixTime'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Pacific/Tongatapu') #
    df = df.sort_values(by="unixTime")

    # Get S/N....
    with open(filename+".TXT", 'r') as file:
        lines = file.readlines()
        if len(lines) > 1:
            # Get the second line and strip the trailing newline character
            SN = lines[1].strip('\n')
            print(SN)
        else:
            print("File has fewer than two lines.")
    

    suffix = "_" +SN[11:15]
    df.columns = [
        f"{col}{suffix}" if col != 'Datetime' else col 
        for col in df.columns
    ]
    dfs.append(df)

# Concatenate all the things:
print('Concatenating:')
print(dfs)
df = pd.concat(dfs, ignore_index=True)

# Localize datetime to Tonga, assuming computer RTC is in Pacific Time:
# df['Datetime'] = df['Datetime'].dt.tz_localize('Pacific/Tongatapu')

# Calculate solar zenith angle:
times = df['Datetime']
solpos = get_solarposition(times, -21, -175) # lat, lon
df['zenith'] = solpos['zenith'].values
df['elevation'] = solpos['elevation'].values
df['azimuth'] = solpos['azimuth'].values
# Filter to time range of experiment:
# df = df[df['Datetime'].between('2026-03-21', '2026-03-23')]
df = df.sort_values('Datetime') # sort to avoid plotting errors

# Sanity check: This should be noonish.
peak_sun = df.loc[df['zenith'].idxmin()]
print(f"Sun is highest at: {peak_sun['Datetime']}")

# Add GPS information
df = df.sort_values('Datetime')
gdf = gdf.sort_values('Datetime')
df = pd.merge_asof(df, gdf, on='Datetime', direction='backward')

# Save dataframe
df.to_csv('ISAC3.csv', index=False)
df.columns()